In [1]:
import nltk

def ngrams(sentenc, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)
sentence = '안녕하세요. 만나서 진심으로 반가워요.'

unigram = ngrams(sentence, 1)
bigram = ngrams(sentence, 2)
trigram = ngrams(sentence, 2)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))

[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]


In [2]:
# 벡터화

from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "That movie is famous movie",
    "I like that actor",
    "I don't like that actor"
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [3]:
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_dim
        )
        self.linear = nn.Linear(
            in_features = embedding_dim,
            out_features = vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [4]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\use\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\use\K

In [5]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [6]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus = tokens, n_vocab = 5000, special_tokens = ['<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [7]:
def get_word_pairs(tokens, window_size):
    pairs = []

    for sentence in tokens:
        sentence_length = len(sentence)

        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)

            context_words = (
                sentence[window_start:idx]
                + sentence[idx + 1:window_end]
            )

            for context_word in context_words:
                pairs.append([center_word, context_word])

    return pairs


word_pairs = get_word_pairs(tokens, window_size=2)
print(len(word_pairs))
print(word_pairs[:5])

2589946
[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [8]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs


index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs_tensor = torch.tensor(index_pairs)
center_indexes = index_pairs_tensor[:, 0]
context_indexes = index_pairs_tensor[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)


In [10]:
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

word2vec = VanillaSkipgram(vocab_size = len(token_to_id), embedding_dim = 128).to(device)

criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)


cuda


In [11]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss
    cost = cost / len(dataloader)
    print(epoch + 1, cost)


1 tensor(6.1981, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(5.9812, device='cuda:0', grad_fn=<DivBackward0>)
3 tensor(5.9314, device='cuda:0', grad_fn=<DivBackward0>)
4 tensor(5.9012, device='cuda:0', grad_fn=<DivBackward0>)
5 tensor(5.8792, device='cuda:0', grad_fn=<DivBackward0>)
6 tensor(5.8616, device='cuda:0', grad_fn=<DivBackward0>)
7 tensor(5.8468, device='cuda:0', grad_fn=<DivBackward0>)
8 tensor(5.8340, device='cuda:0', grad_fn=<DivBackward0>)
9 tensor(5.8225, device='cuda:0', grad_fn=<DivBackward0>)
10 tensor(5.8122, device='cuda:0', grad_fn=<DivBackward0>)


In [12]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[ 0.05697353 -0.41311404  1.1879078   0.5237809   0.4976836  -0.85423696
  1.0123547  -0.20503382 -0.05873994  2.1332147   0.5182632  -0.61057097
 -0.9069514   0.17785154  1.0014608   1.2907903  -0.23173796  2.0459669
  1.0844346   1.4575953  -0.38308063  1.7838384  -0.91450936 -0.49091798
 -1.5089716  -0.681208   -0.314261   -0.08562236  0.08729282 -0.09586465
 -0.35038778  0.62543535  0.29260647  1.5698533   0.57362705  1.2206848
  0.588028   -1.163728    0.89153314 -1.3056203   0.32139435 -0.04463353
  1.1170487  -0.29022378 -0.64621454  0.70160747  0.27042225 -0.4933453
 -1.0656303  -0.26545942 -0.7110074  -1.6985127   0.9762113  -0.03300706
  0.52803046 -0.2920408   0.41407374  0.3493685   1.5143036  -1.080483
 -0.02929358 -1.5215718   0.12566848  0.95538086  0.6899126   1.15531
  1.2909398  -1.1577048  -1.6096894  -0.76759297  0.53811485 -0.96896434
  0.8269147   0.15527569  0.40909776  0.12325139 -0.02466282 -0.29908314
 -0.0230944  -2.6756134   0.78697634  1.0400083   0.8141

In [13]:
import numpy as np
from numpy.linalg import norm


def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1 : n + 1]
    return top_n


cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
로빈 - 유사도 : 0.3080
자연 - 유사도 : 0.2856
들었음 - 유사도 : 0.2777
인해 - 유사도 : 0.2774
느낀다 - 유사도 : 0.2651


In [14]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\use\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\use\K

In [15]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]

In [16]:
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences = tokens,
    vector_size  = 128,
    window = 5,
    min_count = 1,
    sg = 1,
    epochs = 3,
    max_final_vocab = 10000
)

In [17]:
word2vec.save('./models/word2vec.model')
word2vec = Word2Vec.load('./models/word2vec.model')

In [18]:
word = '연기'
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn = 5))
print(word2vec.wv.similarity(w1 = word, w2 = '연기력'))

[-0.40669432 -0.30786988  0.33215344  0.17238484  0.13190204 -0.23395388
  0.14382185 -0.01186585 -0.6300701   0.37296888  0.28480327 -0.4815586
 -0.3173564   0.0334782   0.09402772 -0.10893491 -0.4149054   0.19464536
 -0.0171586   0.01961171  0.55994105  0.00708364 -0.19331713 -0.14556445
 -0.11582108 -0.15006308 -0.1349054   0.10664565  0.20567012 -0.04951525
 -0.5103482  -0.05512656  0.5774895  -0.01265033 -0.1256215   0.30121416
  0.3272024  -0.07760414  0.05922981 -0.11437894  0.07507043  0.1826727
 -0.12722082 -0.38993165 -0.39199778  0.11632993 -0.45343018 -0.08941446
  0.1624885   0.18059787  0.6334299   0.2758717  -0.00116693  0.3558288
 -0.3633461  -0.3926749  -0.22841196 -0.18010433 -0.2053918   0.29755372
 -0.18990692 -0.29246157  0.20339032  0.28872567 -0.41449866  0.0772346
 -0.04146342  0.43837708  0.416737   -0.17495742 -0.23779015 -0.2998663
 -0.19698703  0.08873662 -0.16585279 -0.18939613 -0.23866616 -0.39523867
 -0.3174065   0.12258443 -0.45793325 -0.13256836  0.2602

In [19]:
from Korpora import Korpora

corpus = Korpora.load('kornli')
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

In [20]:
from gensim.models import FastText

fastText = FastText(
    sentences = tokens,
    vector_size = 128,
    window = 5,
    min_count = 5,
    sg = 1,
    max_final_vocab = 20000,
    epochs = 3,
    min_n = 2,
    max_n = 6
)

In [21]:
oov_token = '사랑해요'
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn = 5))


False
[('사랑', 0.8872332572937012), ('사랑에', 0.8252412676811218), ('사랑의', 0.7977014780044556), ('사랑을', 0.7596175670623779), ('사랑하는', 0.7432860732078552)]


In [22]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [23]:
input_size = 128
output_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(input_size = input_size,
               hidden_size = output_size,
               num_layers = num_layers,
               nonlinearity = "tanh",
               batch_first = True,
               bidirectional = bidirectional).to(device)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size).to(device)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 output_size).to(device)
outputs, hidden = model(inputs, h_0)

print(outputs.shape)
print(hidden.shape)
print(outputs.device)


torch.Size([4, 6, 512])
torch.Size([6, 4, 256])
cuda:0


In [24]:
input_size = 128
output_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size = input_size,
    hidden_size = output_size,
    num_layers = num_layers,
    batch_first = True,
    bidirectional = bidirectional,
    proj_size = proj_size
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional) + 1),
    batch_size,
    proj_size if proj_size > 0 else output_size
)

c_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


c:\Users\use\Documents\KDT17\cv-deep-learning\.venv\Lib\site-packages\torch\nn\modules\rnn.py:1124: UserWarning: LSTM with projections is not supported with oneDNN. Using default implementation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\RNN.cpp:1474.)
  result = _VF.lstm(


In [25]:
# 문장분류 실습

import torch.nn as nn

class SentenceClassifier(nn.Module):
    def __init__(self,
                 n_vocab,
                 hidden_dim,
                 embedding_dim,
                 n_layers,
                 dropout = 0.5,
                 bidirectional = True,
                 model_type = "lstm"):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings = n_vocab,
                                      embedding_dim = embedding_dim,
                                      padding_idx = 0)
        if model_type == "rnn":
            self.model = nn.RNN(input_size = embedding_dim,
                                hidden_size = hidden_dim,
                                num_layers = n_layers,
                                bidirectional = bidirectional,
                                dropout = dropout,
                                batch_first = True)
        elif model_type == "lstm":
            self.model = nn.LSTM(input_size = embedding_dim,
                                 hidden_size = hidden_dim,
                                 num_layers = n_layers,
                                 bidirectional = bidirectional,
                                 dropout = dropout,
                                 batch_first = True)

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

In [26]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load('nsmc')
corpus_df = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\use\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\use\K

In [27]:
!pip install tabulate


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\use\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [29]:
train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

# 인덱스 재정렬
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

print(train.head().to_markdown())
print(len(train))
print(len(test))

|    | text                                                                                     |   label |
|---:|:-----------------------------------------------------------------------------------------|--------:|
|  0 | 모든 편견을 날려 버리는 가슴 따뜻한 영화. 로버트 드 니로, 필립 세이모어 호프만 영원하라. |       1 |
|  1 | 무한 리메이크의 소재. 감독의 역량은 항상 그 자리에...                                    |       0 |
|  2 | 신날 것 없는 애니.                                                                       |       0 |
|  3 | 잔잔 격동                                                                                |       1 |
|  4 | 오랜만에 찾은 주말의 명화의 보석                                                         |       1 |
45000
5000


In [32]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenize = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus = train_tokens, n_vocab = 5000, special_tokens = ['<pad>', '<unk>'])

token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

In [33]:
import numpy as np

def pad_sequence(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)
    return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [[token_to_id.get(token, unk_id) for token in review] for review in train_tokens]
test_ids = [[token_to_id.get(token, unk_id) for token in review] for review in test_tokens]

max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequence(train_ids, max_length, pad_id)
test_ids = pad_sequence(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [34]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype = torch.float32)
test_labels = torch.tensor(test.label.values, dtype = torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

print(len(train_loader))
print(len(test_loader))

2813
313


In [37]:
import torch.optim as optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

classifier = SentenceClassifier(
    n_vocab = n_vocab, hidden_dim = hidden_dim, embedding_dim = embedding_dim,
    n_layers = n_layers).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr = 0.001)

In [38]:
def train(model, datasets, criterion, optimizer, device, interval):
    model.train()
    losses = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval == 0:
            print(f"Train Loss {step} : {np.mean(losses)}")


def test(model, datasets, criterion, device):
    model.eval()
    losses = list()
    corrects = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits)>.5
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )

    print(np.mean(losses), np.mean(corrects))

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier,
          train_loader,
          criterion,
          optimizer,
          device,
          interval)
    test(classifier,
         test_loader,
         criterion,
         device)


Train Loss 0 : 0.6919227838516235
Train Loss 500 : 0.6938332954566636
Train Loss 1000 : 0.6820152994397876
Train Loss 1500 : 0.6716401601218923
Train Loss 2000 : 0.6608093544758897
Train Loss 2500 : 0.6519396478106908
0.5663269907712175 0.7198
Train Loss 0 : 0.656914234161377
Train Loss 500 : 0.5380341082870841
Train Loss 1000 : 0.5340243113356513
Train Loss 1500 : 0.530693922487201
Train Loss 2000 : 0.5217862823705325
Train Loss 2500 : 0.5126723756090921
0.48970507361447085 0.7664
Train Loss 0 : 0.504663348197937
Train Loss 500 : 0.4268510271599907
Train Loss 1000 : 0.4196404880145332
Train Loss 1500 : 0.4196974111796617
Train Loss 2000 : 0.4189741857040828
Train Loss 2500 : 0.41729225494155214
0.4183240568580719 0.8038
Train Loss 0 : 0.29025524854660034
Train Loss 500 : 0.36097786334638826
Train Loss 1000 : 0.3639279931381747
Train Loss 1500 : 0.36983134516630867
Train Loss 2000 : 0.36741766019322647
Train Loss 2500 : 0.3698600669030617
0.4081857975679465 0.8172
Train Loss 0 : 0.3303

In [39]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
    token_to_embedding[word] = emb

token = vocab[1000]

print(token, token_to_embedding[token])

보고싶다 [-2.0603325  -1.5308155  -0.04128627  0.37864706 -1.026625   -0.5417257
 -0.3592414   0.5888483   1.1651306  -1.1141324  -2.196554    0.81884116
  1.68462    -1.4291743   0.7534822  -0.5358483  -0.24222727 -1.0921285
 -0.3938451   0.01785808 -0.4950708  -0.3509553  -0.5154008   2.2992175
  0.6443503  -0.20651652  1.6135781  -0.00373258 -0.07861312  0.06120813
 -0.3958103   0.23185699 -1.372891    0.10093995 -1.5256351   1.6276549
 -0.77186435  0.21631435  0.58293474 -0.46471226  1.9193765   1.1813402
  0.83008474 -0.26560986 -0.75142664  0.24641436  1.4049183  -1.6379805
 -0.88476837 -0.8678218  -1.7355115  -0.2778274  -2.2857563   0.10504229
 -1.269137   -1.5431002  -0.914292   -0.7810387   0.51196927  0.83297336
  0.05579581 -0.7027416   1.3168316   0.2708726   2.3877673  -0.2444929
  0.11021959 -0.0311193  -0.71100354  0.69264245  0.13729165  1.3194811
 -0.18676011  1.6937001  -0.94511384 -0.7353419   1.7819946  -0.3577925
 -0.12457351  1.3494753  -1.0295063   1.2705578  -0.084